# Lesson 28: Bayesian & Probabilistic ML

*ML Academy — companion notebook*

Priors and posteriors, Bayesian A/B tests, Bayesian linear regression, Gaussian processes, MCMC, and Thompson sampling.

Run each cell in order with **Shift+Enter**. The explanations, maths and worked examples are in `lesson-28.html`; this notebook contains every code example from the lesson so you can run and modify it.

Requirements: `pip install numpy pandas scikit-learn matplotlib`

## 3. A Bayesian A/B test

In [1]:
import numpy as np
from scipy import stats

# A/B test the Bayesian way: Beta(1, 1) priors, then the observed conversions
a_conv, a_n, b_conv, b_n = 120, 2400, 156, 2400
post_a = stats.beta(1 + a_conv, 1 + a_n - a_conv)
post_b = stats.beta(1 + b_conv, 1 + b_n - b_conv)
rng = np.random.default_rng(0)
da, db = post_a.rvs(200000, random_state=rng), post_b.rvs(200000, random_state=rng)
print("95% credible interval, A:", np.round(post_a.ppf([0.025, 0.975]), 4))
print("95% credible interval, B:", np.round(post_b.ppf([0.025, 0.975]), 4))
print("probability that B converts better than A:", round(np.mean(db > da), 4))
print("expected lift if we switch to B:", round(np.mean(db - da), 4), "  risk (expected loss if B is actually worse):", round(np.mean(np.maximum(da - db, 0)), 5))

95% credible interval, A: [0.042  0.0595]
95% credible interval, B: [0.0558 0.0756]
probability that B converts better than A: 0.9869
expected lift if we switch to B: 0.015   risk (expected loss if B is actually worse): 3e-05


## 5. Bayesian linear regression

In [2]:
# Bayesian linear regression: a normal prior on the weights gives a full posterior, not just one line
x = np.array([0.5, 1.0, 1.5, 3.0, 3.5])
y = np.array([1.1, 1.9, 3.2, 5.8, 7.1])
X = np.column_stack([np.ones_like(x), x])
sigma2, tau2 = 0.25, 10.0                                   # noise variance and prior variance of each weight
S_post = np.linalg.inv(X.T @ X / sigma2 + np.eye(2) / tau2)  # posterior covariance
m_post = S_post @ X.T @ y / sigma2                          # posterior mean (= ridge with lambda = sigma2/tau2)
print("posterior mean (intercept, slope):", m_post.round(3))
print("posterior std  (intercept, slope):", np.sqrt(np.diag(S_post)).round(3))
for x_new in [2.0, 6.0]:
    phi = np.array([1, x_new])
    mean, var = phi @ m_post, phi @ S_post @ phi + sigma2
    print(f"x {x_new}: predicted y {mean:.2f}, 95% prediction interval [{mean - 1.96 * np.sqrt(var):.2f}, {mean + 1.96 * np.sqrt(var):.2f}]")

posterior mean (intercept, slope): [0.086 1.965]
posterior std  (intercept, slope): [0.425 0.192]
x 2.0: predicted y 4.02, 95% prediction interval [2.94, 5.09]
x 6.0: predicted y 11.88, 95% prediction interval [10.00, 13.75]


## 6. Gaussian processes

In [3]:
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, WhiteKernel

# A Gaussian process: uncertainty grows away from the data
xs = np.array([1, 2, 3, 6, 7, 8.0])[:, None]
ys = np.sin(xs[:, 0]) + np.random.default_rng(2).normal(0, 0.1, 6)
gp = GaussianProcessRegressor(RBF(1.0) + WhiteKernel(0.01), random_state=0).fit(xs, ys)
for q in [2.0, 4.5, 11.0]:
    m, s = gp.predict([[q]], return_std=True)
    print(f"x {q:4}: mean {m[0]:+.2f}, std {s[0]:.2f}")

x  2.0: mean +0.86, std 0.00
x  4.5: mean -1.19, std 0.45
x 11.0: mean -0.01, std 0.99


/usr/local/lib/python3.13/site-packages/sklearn/gaussian_process/kernels.py:445: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified lower bound 1e-05. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(


## 7. When there's no formula: MCMC

In [4]:
# MCMC from scratch: sample a posterior that has no neat formula with the Metropolis algorithm
data = np.array([2.1, 2.9, 3.4, 2.2, 8.5, 3.0, 2.7])        # one suspicious value (8.5)
def log_post(mu):                                          # Cauchy likelihood (robust to outliers) and normal prior
    return np.sum(stats.cauchy.logpdf(data, loc=mu, scale=0.5)) + stats.norm.logpdf(mu, 0, 10)
rng = np.random.default_rng(3)
mu, chain, accepted = 0.0, [], 0
for step in range(20000):
    prop = mu + rng.normal(0, 0.5)                         # propose a small random move
    if np.log(rng.random()) < log_post(prop) - log_post(mu):
        mu, accepted = prop, accepted + 1                  # accept with probability min(1, ratio)
    chain.append(mu)
chain = np.array(chain[2000:])                             # drop the warm-up
print("acceptance rate:", round(accepted / 20000, 2))
print("posterior mean of the centre:", round(chain.mean(), 2), "  95% interval:", np.round(np.percentile(chain, [2.5, 97.5]), 2))
print("ordinary mean of the data, for comparison:", round(data.mean(), 2))

acceptance rate: 0.52
posterior mean of the centre: 2.76   95% interval: [2.2  3.28]
ordinary mean of the data, for comparison: 3.54


## 8. Deciding under uncertainty: Thompson sampling

In [5]:
# Thompson sampling: a bandit that learns which of 3 page designs converts best while it runs
true_rates = np.array([0.04, 0.05, 0.065])
wins, losses = np.ones(3), np.ones(3)                      # Beta(1, 1) prior for each design
rng = np.random.default_rng(4)
shown = np.zeros(3, int)
for visitor in range(10000):
    choice = np.argmax(rng.beta(wins, losses))             # sample a plausible rate for each, show the best
    shown[choice] += 1
    if rng.random() < true_rates[choice]: wins[choice] += 1
    else: losses[choice] += 1
print("visitors shown each design:", shown.tolist())
print("conversions:", int(wins.sum() - 3), "  versus a fixed even split (expected):", round(10000 * true_rates.mean()))

visitors shown each design: [354, 458, 9188]
conversions: 652   versus a fixed even split (expected): 517


## 9. In Python: Bayesian ridge with uncertainty

In [6]:
import numpy as np
import pandas as pd
from sklearn.linear_model import BayesianRidge

df = pd.read_csv("../data/houses.csv")   # download it from Lesson 1
X, y = df[["size_m2", "dist_km", "age_yrs"]], df["price_lakh"]
model = BayesianRidge().fit(X, y)
new = pd.DataFrame({"size_m2": [100, 200], "dist_km": [8, 30], "age_yrs": [5, 1]})
mean, std = model.predict(new, return_std=True)
for (_, h), m, s in zip(new.iterrows(), mean, std):
    print(f"{h.size_m2:.0f} m², {h.dist_km:.0f} km, {h.age_yrs:.0f} yrs: {m:.0f} lakh ± {1.96 * s:.0f} (95%)")

100 m², 8 km, 5 yrs: 79 lakh ± 12 (95%)
200 m², 30 km, 1 yrs: 111 lakh ± 20 (95%)


## 12. Going deeper: prior sensitivity and checking MCMC

In [7]:
from scipy import stats
for visitors, conv in [(50, 3), (500, 30), (5000, 300)]:
    row = []
    for name, a, b in [("flat", 1, 1), ("sceptical", 10, 190), ("optimistic", 30, 170)]:
        post = stats.beta(a + conv, b + visitors - conv)
        row.append(f"{name} {post.mean():.3f}")
    print(f"{visitors:5d} visitors (6% converted):", "   ".join(row))

   50 visitors (6% converted): flat 0.077   sceptical 0.052   optimistic 0.132
  500 visitors (6% converted): flat 0.062   sceptical 0.057   optimistic 0.086
 5000 visitors (6% converted): flat 0.060   sceptical 0.060   optimistic 0.063


In [8]:
import numpy as np
data = np.array([2.9, 3.1, 2.4, 3.3, 2.7, 8.5, 2.8])
def log_post(m):
    return -np.sum(np.log1p((data - m) ** 2)) - m ** 2 / 200       # Cauchy likelihood, wide normal prior
def chain(start, step, n=4000, seed=0):
    rng, m, out = np.random.default_rng(seed), start, []
    for _ in range(n):
        prop = m + rng.normal(0, step)
        if np.log(rng.random()) < log_post(prop) - log_post(m):
            m = prop
        out.append(m)
    return np.array(out[n // 2:])                                    # drop the first half as warm-up
def r_hat(chains):
    c = np.array(chains); n = c.shape[1]
    W, B = c.var(axis=1, ddof=1).mean(), n * c.mean(axis=1).var(ddof=1)
    return np.sqrt(((n - 1) / n * W + B / n) / W)
for step in [0.5, 0.005]:
    chains = [chain(s, step, seed=i) for i, s in enumerate([-10, 0, 10, 20])]
    print("step", step, " chain means", [round(float(c.mean()), 2) for c in chains], " R-hat", round(float(r_hat(chains)), 2))

step 0.5  chain means [2.91, 2.91, 2.95, 2.9]  R-hat 1.0


step 0.005  chain means [-9.8, -0.06, 9.84, 20.19]  R-hat 122.7


## Running example: Chennai Homes

The same 12-house dataset used in every lesson (`data/houses.csv` in the course folder).

In [9]:
import pandas as pd
import numpy as np
df = pd.read_csv("../data/houses.csv")
from scipy import stats
for area, g in df.groupby("area"):
    a, b = 1 + g.sold_fast.sum(), 1 + (1 - g.sold_fast).sum()     # Beta(1, 1) prior
    lo, hi = stats.beta(a, b).ppf([0.05, 0.95])
    print(f"{area:9s} sold fast {g.sold_fast.sum()} of {len(g)}  posterior mean {a / (a + b):.2f}  90% interval [{lo:.2f}, {hi:.2f}]")

Central   sold fast 3 of 4  posterior mean 0.67  90% interval [0.34, 0.92]
Outskirts sold fast 0 of 3  posterior mean 0.20  90% interval [0.01, 0.53]
Suburb    sold fast 3 of 5  posterior mean 0.57  90% interval [0.27, 0.85]


## Coding challenges

Each cell below has **one missing line**, marked `# ← your code here` (or `# ← replace`). Write the line, run the cell, and the check at the bottom tells you whether you got it right. Stuck? A wrong answer prints a hint, and the solutions are at the very end of this notebook.

In [10]:
# Checker used by the challenges below — run this cell first.
import numpy as np
def check(got, want, hint='', atol=1e-6):
    try:
        ok = got is not None and np.shape(got) == np.shape(want) and np.allclose(np.asarray(got, dtype=float), np.asarray(want, dtype=float), rtol=1e-4, atol=atol)
    except Exception:
        ok = False
    print('✅ Correct!' if ok else '❌ Not yet. Hint: ' + hint)


### Challenge 1: A Beta posterior

Update a Beta(2, 2) prior with 7 successes and 3 failures, and compute the posterior mean.

In [11]:
a, b = 2, 2
s, f = 7, 3
mean = s / (s + f)  # ← that ignores the prior
check(mean, 0.6428571428571429, hint='The posterior is Beta(a + s, b + f); its mean is a′/(a′ + b′).')

❌ Not yet. Hint: The posterior is Beta(a + s, b + f); its mean is a′/(a′ + b′).


### Challenge 2: A Metropolis acceptance

Compute the acceptance probability for a proposal.

In [12]:
current, proposal = 0.25, 0.05
accept = proposal  # ← use the ratio, capped at 1
check(accept, 0.2, hint='Ratio of densities, capped at 1.')

❌ Not yet. Hint: Ratio of densities, capped at 1.


## Your turn

Try the **practice exercises** at the end of the lesson page. Many of them ask you to change one of the cells above and re-run it.

## Challenge solutions

*Spoilers — try the challenges first!*

**1. A Beta posterior** — the missing line:

```python
mean = (a + s) / (a + s + b + f)
```

**2. A Metropolis acceptance** — the missing line:

```python
accept = min(1, proposal / current)
```